## 03 - Model Training (Logistic Regression)

This notebook trains a Logistic Regression model using the processed training and test datasets.

It includes:
- Loading `X_train`, `X_test`, `y_train`, and `y_test` from local pickle files or S3 CSVs
- Training a Logistic Regression model using final hyperparameter settings
- Evaluating model performance on the test set
- Saving the trained model locally and uploading to S3 if running in SageMaker

**Note:** The validation set is reserved for later use in monitoring and CI/CD.


In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import boto3
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Sagemaker Setup

In [3]:
from sagemaker import get_execution_role
from sagemaker.session import Session
import boto3

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client("sagemaker", region_name=region)
featurestore_runtime = boto_session.client("sagemaker-featurestore-runtime", region_name=region)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

role = get_execution_role()
bucket = feature_store_session.default_bucket()


## Loading Training Data

In [4]:
# Set this flag to 's3' if running on SageMaker or loading from S3 bucket

prefix = "diabetes/training"

LOAD_MODE = "local"  # options: "local" or "s3"

def load_data(filename):
    if LOAD_MODE == "local":
        return pd.read_pickle(os.path.join("data", filename))
    else:
        s3_uri = f"s3://{bucket}/{prefix}/{filename}"
        return pd.read_csv(s3_uri)

# Load only train and test data
X_train = load_data("X_train.pkl" if LOAD_MODE == "local" else "X_train.csv")
y_train = load_data("y_train.pkl" if LOAD_MODE == "local" else "y_train.csv")
X_test  = load_data("X_test.pkl"  if LOAD_MODE == "local" else "X_test.csv")
y_test  = load_data("y_test.pkl"  if LOAD_MODE == "local" else "y_test.csv")

# Ensure labels are Series
if LOAD_MODE == "s3":
    y_train = y_train.squeeze()
    y_test  = y_test.squeeze()

print("Data loaded:")
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)


Data loaded:
Train: (61059, 189) (61059,)
Test: (20354, 189) (20354,)


## Best Model for Logistic Regression

Refer to FinalProject.ipynb for the model design

In [5]:
# Final Logistic Regression model with selected hyperparameters

# Create a pipeline: StandardScaler + LogisticRegression
best_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        C=0.01,
        class_weight='balanced',
        penalty='l2',
        solver='lbfgs',
        max_iter=1000,
        random_state=42
    ))
])

# Train on full training set
best_model.fit(X_train, y_train)
print("Final Logistic Regression pipeline trained.")


Final Logistic Regression pipeline trained.


## Save Model

In [6]:
import joblib
import os

os.makedirs("model", exist_ok=True)
joblib.dump(best_model, "model/model.joblib")


['model/model.joblib']

In [7]:
import tarfile

# Create the SageMaker-compatible tar.gz package
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model/model.joblib", arcname="model.joblib")  # Flatten to root inside archive

print("model.tar.gz created successfully.")


model.tar.gz created successfully.


In [8]:
from sagemaker.s3 import S3Uploader

model_s3_uri = S3Uploader.upload("model.tar.gz", f"s3://{bucket}/diabetes/model")
print("Uploaded model to:", model_s3_uri)


Uploaded model to: s3://sagemaker-us-east-1-380537322556/diabetes/model/model.tar.gz
